In [2]:
import polars as pl
import os
from pathlib import Path

def process_snow_water_data():
    """
    Process snow water data for specified year and month combinations.
    Creates separate CSVs for each year and month combination.
    """
    current_dir = os.getcwd()
    
    # Get all CSV files in the directory
    all_files = list(Path(current_dir).glob("us_ssmv11034tS__T0001TTNATS*.csv"))
    
    if not all_files:
        print("No matching files found.")
        return

    # Define the target years and months
    target_years = range(2010, 2021)  # 2010 to 2020
    target_months = ['01', '02', '12']  # January, February, December
    
    # Iterate over each year and month combination
    for year in target_years:
        for month in target_months:
            # Filter files for the current year and month
            target_files = [
                f for f in all_files 
                if f.name[27:31] == str(year) and f.name[31:33] == month
            ]
            
            if not target_files:
                print(f"No files found for {year}-{month}")
                continue
            
            print(f"Found {len(target_files)} files for {year}-{month}")
            
            # Initialize list to store valid DataFrames
            valid_dfs = []
            
            # Process each file
            for file_path in target_files:
                try:
                    if file_path.stat().st_size == 0:
                        print(f"Skipping empty file: {file_path.name}")
                        continue
                    
                    df = pl.read_csv(
                        file_path,
                        has_header=True,
                        new_columns=['X', 'Y', 'Value', 'Year', 'Month']
                    )
                    
                    # Replace invalid values with None
                    df = df.with_columns(
                        pl.when(pl.col('Value').is_in([55537, -9999]))
                        .then(None)
                        .otherwise(pl.col('Value'))
                        .alias('Value')
                    )
                    
                    valid_dfs.append(df)
                    print(f"Processed {file_path.name}")
                    
                except Exception as e:
                    print(f"Error processing {file_path.name}: {e}")
                    continue
            
            if not valid_dfs:
                print(f"No valid data found for {year}-{month}")
                continue
            
            # Combine all DataFrames and compute averages
            result = (pl.concat(valid_dfs)
                .group_by(['X', 'Y'])
                .agg(
                    pl.col('Value').mean().alias('Value'),
                    pl.lit(year).alias('Year'),
                    pl.lit(int(month)).alias('Month')
                )
            )
            
            # Write output for the current year and month
            output_file = f"snow_water_equiv_{year}_{month}_avg.csv"
            result.write_csv(output_file)
            print(f"Created {output_file}")
    
    print("Processing complete!")

if __name__ == "__main__":
    process_snow_water_data()

Found 31 files for 2010-01
Processed us_ssmv11034tS__T0001TTNATS2010012705HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010011605HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010012805HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010605HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010505HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010405HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010011105HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010011505HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010012305HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010012905HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010105HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010905HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010010705HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010012005HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010013105HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010012405HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2010011705HP001.csv
Proc

1.8.2
